# Static ASL Alphabet Recognition — MobileNet improvement
**Ramdeobaba University | Deep Learning Lab | 2026–27**

Our first custom CNN reached 86.05% accuracy on the held-out dataset images.
This experiment uses the **MobileNet option in our submitted proposal**.
We keep the same Akash Nagaraj ASL Alphabet dataset, 25 classes, and data splits.

We train a small classifier on MobileNetV3Small's pretrained features, then
fine-tune its convolution layers. The ImageNet weights are the starting point;
our training adapts those features to the ASL classes.

The image size is increased to 160 × 160 to retain more finger detail.
Model selection uses validation accuracy. The test set is reused for a
baseline comparison, so it is not a new independent evaluation dataset.
We do not change the test split or remove difficult classes.

**Before Run All:** attach `grassknoted/asl-alphabet`, enable Internet,
and choose GPU T4 x2. The notebook uses one T4 and stops if no GPU is available.
MediaPipe hand cropping and temporal voting remain in the laptop application.


## 1. Settings and GPU check
The split seed and class list are unchanged. We use one GPU to keep the code simple.
The first training stage learns the classifier; the second adapts pretrained features.


In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import sys, json, time, random, hashlib, re, platform, shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

SEED = 43
IMAGE_SIZE = 160
BATCH_SIZE = 128
WARMUP_EPOCHS = 5
FINE_TUNE_EPOCHS = 25
MAX_EPOCHS = WARMUP_EPOCHS + FINE_TUNE_EPOCHS
USE_TWO_GPUS = False
BLOCK_SIZE = 100
CLASSES = [c for c in 'ABCDEFGHIJKLMNOPQRSTUVWXYZ' if c not in 'JZ'] + ['nothing']
OUT = Path('/kaggle/working/asl_improved_results')
OUT.mkdir(parents=True, exist_ok=True)
keras.utils.set_random_seed(SEED)
gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow:', tf.__version__)
print('Python:', sys.version.split()[0])
print('Visible GPUs:', gpus)
if not gpus:
    raise RuntimeError('GPU required: complete phone verification, select GPU T4 x2, then Run All again.')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
strategy = (tf.distribute.MirroredStrategy() if USE_TWO_GPUS
            else tf.distribute.OneDeviceStrategy('/GPU:0'))
print('Replicas used:', strategy.num_replicas_in_sync)
config = dict(seed=SEED, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
              max_epochs=MAX_EPOCHS, classes=CLASSES, block_size=BLOCK_SIZE,
              two_gpus=USE_TWO_GPUS, dataset='grassknoted/asl-alphabet',
              tensorflow=tf.__version__, python=sys.version, platform=platform.platform())
(OUT/'config.json').write_text(json.dumps(config, indent=2))
config.update(architecture='MobileNetV3Small', initialization='ImageNet pretrained', selection='validation accuracy')


## 2. Locate and inspect the exact dataset
Keep the original Kaggle dataset attached for attribution. For faster reading, download
the same public archive to the session's local disk and check its SHA-256 before use.
This avoids opening thousands of small images through the slower mounted input path.
Turn on Internet in Kaggle Settings. The supplied tiny test folder is not used for evaluation.


In [ ]:
import urllib.request
import zipfile

archive_path = Path('/kaggle/temp/asl-alphabet.zip')
archive_path.parent.mkdir(parents=True, exist_ok=True)
expected_sha = '7c572f14fbaff94f98835cfe71c7582dd379a5176e7c4f83dbf3a30e4b3f68c4'
if not archive_path.exists():
    url = 'https://www.kaggle.com/api/v1/datasets/download/grassknoted/asl-alphabet'
    partial = archive_path.with_suffix('.part')
    started = time.perf_counter()
    with urllib.request.urlopen(url, timeout=120) as response, partial.open('wb') as f:
        downloaded = 0
        while True:
            chunk = response.read(8*1024*1024)
            if not chunk: break
            f.write(chunk)
            downloaded += len(chunk)
            if downloaded//(128*1024*1024) != (downloaded-len(chunk))//(128*1024*1024):
                print(f'Downloaded {downloaded/1e6:.0f} MB', flush=True)
    partial.replace(archive_path)
    print('Download seconds:', round(time.perf_counter()-started, 1))
digest = hashlib.sha256()
with archive_path.open('rb') as f:
    for chunk in iter(lambda: f.read(8*1024*1024), b''):
        digest.update(chunk)
if digest.hexdigest() != expected_sha:
    raise ValueError('Dataset archive checksum changed. Inspect its version before continuing.')
local_dir = Path('/kaggle/temp/asl_data')
ready = local_dir/'extraction_complete.txt'
if not ready.exists():
    with zipfile.ZipFile(archive_path) as z:
        for info in z.infolist():
            names = info.filename.split('/')
            if (len(names) == 4 and names[:2] == ['asl_alphabet_train','asl_alphabet_train']
                    and names[2] in CLASSES and names[3].endswith('.jpg') and '..' not in names):
                z.extract(info, local_dir)
    ready.write_text(expected_sha)
DATA_ROOT = local_dir/'asl_alphabet_train'/'asl_alphabet_train'
config['dataset_archive_sha256'] = expected_sha
(OUT/'config.json').write_text(json.dumps(config, indent=2))
print('Training images:', DATA_ROOT)
rows = []
for label, name in enumerate(CLASSES):
    paths = sorted(p for p in (DATA_ROOT/name).iterdir()
                   if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})
    if len(paths) < 100:
        raise ValueError(f'Unexpected or missing data for {name}: {len(paths)} images')
    for p in paths:
        rows.append(dict(path=str(p), relative_path=str(p.relative_to(DATA_ROOT)),
                         class_name=name, label=label))
df = pd.DataFrame(rows)
display(df.groupby('class_name', sort=False).size().rename('images').to_frame())
print('Selected images:', len(df), '| Output classes:', len(CLASSES))
fig, axes = plt.subplots(5, 5, figsize=(10, 10))
for ax, name in zip(axes.flat, CLASSES):
    p = df.loc[df.class_name.eq(name), 'path'].iloc[0]
    with Image.open(p) as im:
        ax.imshow(im.convert('RGB'))
    ax.set_title(name); ax.axis('off')
plt.tight_layout(); plt.savefig(OUT/'class_examples.png', dpi=150); plt.show()


## 3. Audit exact duplicates and create reproducible splits
Hash files before splitting. Keep one copy of each exact duplicate; remove conflicting
labels and retain an audit CSV. To reduce leakage from consecutive captures, group
each class's numerically ordered filenames into blocks of 100 before assigning blocks
to train/validation/test (approximately 70/15/15, depending on available blocks).

**Limitation:** filename blocks are a proxy, not verified recording sessions or subjects.
The dataset does not provide reliable signer identities here, so this is not a
subject-independent split. Similar poses/backgrounds can remain across blocks.
The separate volunteer evaluation is needed for claims about new people.


In [ ]:
def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

audit_start = time.perf_counter()
with ThreadPoolExecutor(max_workers=8) as pool:
    df['sha256'] = list(pool.map(sha256_file, df.path))
conflicts = df.groupby('sha256').class_name.nunique()
bad_hashes = set(conflicts[conflicts > 1].index)
audit = df[df.duplicated('sha256', keep=False)].copy()
audit['conflicting_labels'] = audit.sha256.isin(bad_hashes)
audit.to_csv(OUT/'duplicate_audit.csv', index=False)
original_count = len(df)
df = df[~df.sha256.isin(bad_hashes)].drop_duplicates('sha256').copy()

def numeric_index(path):
    matches = re.findall(r'\d+', Path(path).stem)
    if not matches:
        raise ValueError('Expected numbered filenames; inspect the dataset before splitting.')
    return int(matches[-1])

df['capture_block'] = df.path.map(numeric_index) // BLOCK_SIZE
df['split'] = ''
rng = np.random.default_rng(SEED)
for name in CLASSES:
    mask = df.class_name.eq(name)
    groups = np.array(sorted(df.loc[mask, 'capture_block'].unique()))
    if len(groups) < 7:
        raise ValueError(f'Not enough blocks for {name}; inspect BLOCK_SIZE.')
    rng.shuffle(groups)
    n_test = max(1, round(len(groups)*0.15))
    n_val = max(1, round(len(groups)*0.15))
    test_groups, val_groups = groups[:n_test], groups[n_test:n_test+n_val]
    df.loc[mask, 'split'] = 'train'
    df.loc[mask & df.capture_block.isin(val_groups), 'split'] = 'val'
    df.loc[mask & df.capture_block.isin(test_groups), 'split'] = 'test'

assert df.groupby(['class_name', 'capture_block']).split.nunique().max() == 1
assert df.sha256.is_unique
assert not df.split.eq('').any()
split_summary = pd.crosstab(df.class_name, df.split).reindex(CLASSES)
assert (split_summary[['train', 'val', 'test']] > 0).all().all()
display(split_summary)
df.to_csv(OUT/'split_manifest.csv', index=False)
split_summary.to_csv(OUT/'split_summary.csv')
print('Removed duplicates/conflicts:', original_count-len(df))
print('Audit seconds:', round(time.perf_counter()-audit_start, 1))

# Confirm that every image keeps its original train/validation/test assignment.
split_text = df[['relative_path','label','split']].sort_values('relative_path').to_csv(index=False, lineterminator='\n')
split_digest = hashlib.sha256(split_text.encode()).hexdigest()
assert split_digest == '1d1ec94f59e832a7e28649798835dd016b832c79cfc0f8373910310b3fe9b564', 'Data splits differ from the baseline.'
config['split_sha256'] = split_digest
(OUT/'config.json').write_text(json.dumps(config, indent=2))
print('Original data splits verified.')


## 4. Image loading
Decode RGB images and resize with padding to 160 × 160. Cache uint8 images in RAM.
Only training images are shuffled and augmented. MobileNet includes its input
rescaling, so we supply raw RGB values in [0,255] during training and inference.


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
def decode_image(path, label):
    raw = tf.io.read_file(path)
    image = tf.io.decode_image(raw, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize_with_pad(image, IMAGE_SIZE, IMAGE_SIZE)
    return tf.cast(tf.clip_by_value(image, 0, 255), tf.uint8), label

parts = {s: df[df.split.eq(s)].sort_values(['label', 'relative_path']).reset_index(drop=True)
         for s in ['train', 'val', 'test']}
def make_dataset(frame, training=False):
    ds = tf.data.Dataset.from_tensor_slices((frame.path.to_numpy(), frame.label.to_numpy(dtype=np.int32)))
    ds = ds.map(decode_image, num_parallel_calls=AUTOTUNE).cache()
    if training:
        ds = ds.shuffle(min(len(frame), 10000), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(parts['train'], True)
val_ds = make_dataset(parts['val'])
test_ds = make_dataset(parts['test'])
images, labels = next(iter(train_ds.take(1)))
print('Batch:', images.shape, images.dtype, '| labels:', labels.shape)


## 5. MobileNetV3Small and a simple classifier
MobileNet is a compact CNN. Its ImageNet weights give us a useful starting point
for edges, textures, and shapes. Global average pooling produces a feature vector,
followed by a dense layer, dropout, and 25 output scores.

First we freeze the backbone and train only the classifier. Fine-tuning later uses
a smaller learning rate. Batch-normalization statistics remain fixed for stability.
We keep the mild augmentation and avoid horizontal flips.


In [ ]:
with strategy.scope():
    augmentation = keras.Sequential([
        layers.RandomRotation(0.03, fill_mode='constant', fill_value=0., seed=SEED),
        layers.RandomTranslation(0.06, 0.06, fill_mode='constant', fill_value=0., seed=SEED+1),
        layers.RandomZoom(0.08, fill_mode='constant', fill_value=0., seed=SEED+2),
        layers.RandomContrast(0.1, seed=SEED+3),
    ], name='training_augmentation')
    base = keras.applications.MobileNetV3Small(
        input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), include_top=False,
        weights='imagenet', include_preprocessing=True)
    base.trainable = False
    base_name = base.name
    inputs = keras.Input((IMAGE_SIZE, IMAGE_SIZE, 3), name='rgb_0_to_255')
    x = augmentation(inputs)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3, seed=SEED+4)(x)
    outputs = layers.Dense(len(CLASSES), activation='softmax', dtype='float32')(x)
    model = keras.Model(inputs, outputs, name='static_asl_mobilenet')
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()
with (OUT/'model_summary.txt').open('w') as f:
    model.summary(print_fn=lambda line: f.write(line+'\n'))
(OUT/'model_architecture.json').write_text(model.to_json())


## 6. Train the classifier, then fine-tune
We choose the best checkpoint using validation accuracy, before looking at test results.
Warm-up trains only the new classifier. Fine-tuning updates the convolution weights
with a lower learning rate, while batch-normalization layers stay frozen.

Epoch logs and both phase checkpoints are saved. The final selected model is exported
without optimizer state. Save the executed Kaggle version with outputs to keep the files.


In [ ]:
epoch_seconds = []
class EpochTimer(keras.callbacks.Callback):
    def on_epoch_begin(self, epoch, logs=None):
        self.started = time.perf_counter()
    def on_epoch_end(self, epoch, logs=None):
        seconds = time.perf_counter()-self.started
        epoch_seconds.append(seconds)
        pd.DataFrame({'epoch': range(1,len(epoch_seconds)+1), 'seconds': epoch_seconds}).to_csv(OUT/'epoch_times.csv', index=False)
        print(f'  epoch wall time: {seconds:.1f}s')

def training_callbacks(phase):
    return [
        keras.callbacks.ModelCheckpoint(str(OUT/f'best_{phase}.keras'), monitor='val_accuracy', mode='max', save_best_only=True),
        keras.callbacks.CSVLogger(str(OUT/f'{phase}_log.csv')),
        keras.callbacks.EarlyStopping(monitor='val_accuracy', mode='max', patience=6, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6),
        keras.callbacks.TerminateOnNaN(), EpochTimer(),
    ]

training_started = time.perf_counter()
print('STAGE 1: train the classifier')
warm = model.fit(train_ds, validation_data=val_ds, epochs=WARMUP_EPOCHS,
                 callbacks=training_callbacks('head'), verbose=2)

with strategy.scope():
    model = keras.models.load_model(OUT/'best_head.keras', compile=False)
    base = model.get_layer(base_name)
    base.trainable = True
    for layer in base.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    model.compile(optimizer=keras.optimizers.Adam(1e-4),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
print('STAGE 2: fine-tune convolution layers')
fine = model.fit(train_ds, validation_data=val_ds, epochs=FINE_TUNE_EPOCHS,
                 callbacks=training_callbacks('finetuned'), verbose=2)
training_seconds = time.perf_counter()-training_started
head_best = max(warm.history['val_accuracy'])
fine_best = max(fine.history['val_accuracy'])
selected_phase = 'finetuned' if fine_best >= head_best else 'head'
selected_val_accuracy = max(head_best, fine_best)
model = keras.models.load_model(OUT/f'best_{selected_phase}.keras', compile=False)
model.save(OUT/'best_model.keras')
warm_df, fine_df = pd.DataFrame(warm.history), pd.DataFrame(fine.history)
warm_df['phase'], fine_df['phase'] = 'head', 'fine_tuning'
history_df = pd.concat([warm_df,fine_df], ignore_index=True)
history_df.index.name = 'epoch_zero_based'
history_df.to_csv(OUT/'history.csv')
print('Selected phase:', selected_phase)
print('Best validation accuracy:', selected_val_accuracy)
print('Training seconds:', round(training_seconds, 1))


## 7. Learning curves and held-out dataset evaluation
This test uses unseen filename blocks from the same public dataset, not unseen signers.
Do not tune parameters using these test results and still describe the set as untouched.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for key in ['accuracy', 'loss']:
    ax = axes[0 if key == 'accuracy' else 1]
    ax.plot(history_df[key], label='train')
    ax.plot(history_df['val_'+key], label='validation')
    ax.set(xlabel='Epoch (zero based)', ylabel=key, title=key.title()); ax.legend(); ax.grid(alpha=.2)
plt.tight_layout(); plt.savefig(OUT/'learning_curves.png', dpi=180); plt.show()

scores = model.predict(test_ds, verbose=1)
y_true = parts['test'].label.to_numpy()
y_pred = scores.argmax(axis=1)
metrics = dict(test_accuracy=float(accuracy_score(y_true, y_pred)),
               test_macro_f1=float(f1_score(y_true, y_pred, labels=range(len(CLASSES)), average='macro', zero_division=0)),
               training_seconds=training_seconds, epochs_completed=len(history_df),
               parameters=model.count_params(), test_images=len(y_true),
               split_type='exact-deduplicated filename-block split; not subject-independent')
metrics.update(architecture='MobileNetV3Small', initialization='ImageNet pretrained',
               selected_phase=selected_phase, selected_validation_accuracy=float(selected_val_accuracy),
               checkpoint_monitor='validation accuracy', test_reused_for_baseline_comparison=True)
report = classification_report(y_true, y_pred, labels=range(len(CLASSES)), target_names=CLASSES,
                               output_dict=True, zero_division=0)
pd.DataFrame(report).T.to_csv(OUT/'classification_report.csv')
print(json.dumps(metrics, indent=2))
display(pd.DataFrame(report).T)
predictions = parts['test'].copy()
predictions['prediction'] = [CLASSES[i] for i in y_pred]
predictions['score'] = scores.max(axis=1)
predictions.to_csv(OUT/'test_predictions.csv', index=False)
cm = confusion_matrix(y_true, y_pred, labels=range(len(CLASSES)))
fig, ax = plt.subplots(figsize=(14, 12))
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(ax=ax, xticks_rotation=90, colorbar=False,
                                                      values_format='d', cmap='Blues')
ax.set_title('Held-out public-dataset images (not unseen signers)')
plt.tight_layout(); plt.savefig(OUT/'confusion_matrix.png', dpi=170); plt.show()
confused = [(CLASSES[i], CLASSES[j], int(cm[i,j])) for i in range(len(CLASSES))
            for j in range(len(CLASSES)) if i != j and cm[i,j] > 0]
display(pd.DataFrame(sorted(confused, key=lambda x: -x[2])[:15],
                     columns=['Actual', 'Predicted', 'Count']))

baseline_metrics = {'test_accuracy': 0.860515923031678, 'test_macro_f1': 0.8594449968351744, 'training_seconds': 168.86819780899998, 'epochs_completed': 16, 'parameters': 1206585, 'test_images': 11901, 'split_type': 'exact-deduplicated filename-block split; not subject-independent', 'cloud_model_only_latency_median_ms': 0.940919499953452, 'cloud_model_only_latency_p95_ms': 1.0371633499971722, 'cloud_model_only_fps': 1052.3866719575813}
comparison = pd.DataFrame([{'model':'Original CNN','test_accuracy':baseline_metrics['test_accuracy'],'macro_f1':baseline_metrics['test_macro_f1']}, {'model':'MobileNetV3Small','test_accuracy':metrics['test_accuracy'],'macro_f1':metrics['test_macro_f1']}])
comparison.to_csv(OUT/'baseline_comparison.csv',index=False)
display(comparison)


## 8. Inspect mistakes and measure model-only speed
Visual error analysis is part of the project. The following latency uses batch size 1
on this cloud device, after warm-up. It excludes webcam capture, hand detection,
display, and temporal voting, so it must not be reported as end-to-end webcam FPS.


In [ ]:
wrong = np.flatnonzero(y_true != y_pred)[:12]
if len(wrong):
    fig, axes = plt.subplots(3, 4, figsize=(12, 9))
    for ax in axes.flat: ax.axis('off')
    for ax, idx in zip(axes.flat, wrong):
        with Image.open(parts['test'].path.iloc[idx]) as im:
            ax.imshow(im.convert('RGB'))
        ax.set_title(f'True: {CLASSES[y_true[idx]]} | Pred: {CLASSES[y_pred[idx]]}')
    plt.tight_layout(); plt.savefig(OUT/'error_examples.png', dpi=160); plt.show()

sample = tf.cast(images[:1], tf.float32)
@tf.function
def predict_one(x):
    return model(x, training=False)
for _ in range(20): predict_one(sample).numpy()
latencies = []
for _ in range(100):
    t0 = time.perf_counter()
    predict_one(sample).numpy()  # synchronize GPU result
    latencies.append((time.perf_counter()-t0)*1000)
metrics['cloud_model_only_latency_median_ms'] = float(np.median(latencies))
metrics['cloud_model_only_latency_p95_ms'] = float(np.percentile(latencies, 95))
metrics['cloud_model_only_fps'] = float(1000/np.mean(latencies))
(OUT/'metrics.json').write_text(json.dumps(metrics, indent=2))
print('Model-only benchmark:', {k:v for k,v in metrics.items() if 'model_only' in k})


## 9. Optional separate volunteer evaluation
Do not use final evaluation volunteers/recording sessions for training or tuning.
Attach a separate dataset with `volunteer_test/A/*.jpg`, etc., and edit the path below.
It must use the same MediaPipe cropping/preprocessing as the webcam application.
Until such images are supplied, volunteer performance is explicitly **not measured**.


In [ ]:
VOLUNTEER_ROOT = None  # e.g. Path('/kaggle/input/our-held-out-signs/volunteer_test')
if VOLUNTEER_ROOT is None:
    print('Volunteer evaluation pending. No local-user accuracy is claimed.')
else:
    vroot = Path(VOLUNTEER_ROOT)
    unknown = sorted(p.name for p in vroot.iterdir() if p.is_dir() and p.name not in CLASSES)
    if unknown: raise ValueError(f'Unknown class folders: {unknown}')
    local_rows = [dict(path=str(p), label=i) for i, name in enumerate(CLASSES)
                  for p in sorted((vroot/name).glob('*')) if p.suffix.lower() in {'.jpg','.jpeg','.png'}]
    if not local_rows: raise ValueError('No volunteer images found.')
    local_df = pd.DataFrame(local_rows)
    local_scores = model.predict(make_dataset(local_df), verbose=1)
    local_pred = local_scores.argmax(1)
    local_report = classification_report(local_df.label, local_pred, labels=range(len(CLASSES)),
                                         target_names=CLASSES, output_dict=True, zero_division=0)
    pd.DataFrame(local_report).T.to_csv(OUT/'volunteer_classification_report.csv')
    print('Volunteer accuracy:', accuracy_score(local_df.label, local_pred))
    print('Macro-F1 includes absent classes; inspect support in the report.')
    display(pd.DataFrame(local_report).T)


## 10. Export for the webcam application
Use `best_model.keras` and `inference_config.json` together. The model includes
MobileNet's rescaling and expects RGB values in [0,255]. Do not normalize again.
The webcam code reads the new input size from this configuration automatically.

The existing MediaPipe crop and 7-frame/5-vote smoothing remain unchanged.
Local volunteer and webcam testing are still required; public-dataset accuracy
does not establish performance on unseen people or unknown signs.


In [ ]:
inference_config = dict(classes=CLASSES, image_size=IMAGE_SIZE,
                        colour_order='RGB', pixel_range=[0,255],
                        resize='aspect-preserving resize_with_pad',
                        score_threshold=0.75, vote_window=7, required_votes=5,
                        hand_crop_margin=0.20, threshold_status='initial heuristic; not calibrated')
(OUT/'inference_config.json').write_text(json.dumps(inference_config, indent=2))
requirements = []
for name in ['tensorflow','keras','numpy','pandas','scikit-learn','Pillow','matplotlib']:
    try:
        from importlib.metadata import version
        requirements.append(f'{name}=={version(name)}')
    except Exception: pass
(OUT/'training_environment.txt').write_text('\n'.join(requirements)+'\n')

(OUT/'MODEL_CARD.md').write_text(
    '# Static ASL MobileNetV3Small\n\n'
    'Dataset: Akash Nagaraj / grassknoted/asl-alphabet.\n'
    '24 static letters plus nothing. Same baseline data splits.\n'
    'ImageNet pretrained backbone, classifier warm-up, then convolution fine-tuning.\n'
    'Selected by validation accuracy. Test reused for baseline comparison.\n'
    'Filename-block test is not unseen-signer evaluation. Volunteer testing remains pending.\n'
    'Inputs: RGB 0-255; rescaling inside model. Scores are uncalibrated.\n'
    'Cloud model-only latency excludes capture, MediaPipe, rendering, and voting.\n'
)

# Check that the exported model keeps the same inputs, outputs, and predictions.
check_images, _ = next(iter(test_ds))
check_images = check_images[:32]
restored = keras.models.load_model(OUT/'best_model.keras', compile=False)
assert restored.input_shape[1:3] == (IMAGE_SIZE, IMAGE_SIZE)
assert restored.output_shape[-1] == len(CLASSES)
before = model(check_images, training=False).numpy()
after = restored(check_images, training=False).numpy()
np.testing.assert_allclose(before, after, rtol=1e-4, atol=1e-6)
check = dict(images_checked=len(check_images), max_score_difference=float(np.max(np.abs(before-after))), status='passed')
(OUT/'export_verification.json').write_text(json.dumps(check, indent=2))
(OUT/'model_architecture.json').write_text(model.to_json())
with (OUT/'selected_model_summary.txt').open('w') as f:
    model.summary(print_fn=lambda line: f.write(line+'\n'))
print('Export verification:', check)
del restored

import zipfile
archive = '/kaggle/working/asl_improved_model_and_results.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for path in OUT.iterdir():
        if path.is_file() and path.name not in {'best_head.keras','best_finetuned.keras'}:
            z.write(path,path.name)
print('Export ready:', archive)
print('Keep this archive and the executed notebook using Save Version.')
print('Actual measured metrics:', json.dumps(metrics, indent=2))


## Project discussion and next checks
- Explain each convolution, pooling, dense, dropout, softmax, loss, and optimizer choice.
- Explain why augmentation is training-only and why test data cannot guide tuning.
- Explain that MediaPipe detects the hand; the CNN classifies the sign.
- Compare raw predictions with 7-frame voting on the same recorded clips. Record
  flicker/change counts and time from a held sign to a stable display; do not invent results.
- Measure webcam FPS and end-to-end latency on the laptop separately from cloud inference.
- Test different volunteers and backgrounds, including failures and no-hand frames.
- Compare the original CNN with MobileNet using the same split and evaluation metrics.

### References
1. Akash Nagaraj, ASL Alphabet, Kaggle, 2018. https://www.kaggle.com/datasets/grassknoted/asl-alphabet
2. TensorFlow image classification: https://www.tensorflow.org/tutorials/images/classification
3. MediaPipe Hands: https://ai.google.dev/edge/mediapipe/solutions/vision/hand_landmarker

MobileNet reference: https://keras.io/api/applications/mobilenet/mobilenet_models/
Transfer learning: https://www.tensorflow.org/tutorials/images/transfer_learning
